# Calibration Raman — Waterkit (FR)
*Généré le 2025-11-04*

Ce notebook charge des **spectres étalons** (eau + solides), applique un **lissage léger** + **baseline polynomiale**, extrait des **aires de bandes**, **centroïdes** et **FWHM**, puis construit des **courbes d'étalonnage** (aire vs concentration). Le code évite les dépendances lourdes (SciPy) et fonctionne avec NumPy pur.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import json, os, sys
base = Path('..').resolve()
sys.path.append(str(base/'scripts'))
import raman_cal_helpers as R
data_dir = base/'data_templates'

## Paramètres d'analyse (fenêtres des bandes)

In [ ]:
windows = {
    'SO4': (970, 992),     # 981 cm-1
    'NO3': (1038, 1056),   # 1047 cm-1
    'CO3': (1055, 1076),   # 1066 cm-1
    'HCO3': (1010, 1024),  # ~1017-1018 cm-1
    'PO4': (928, 948),     # 938 cm-1
    'C6H12_801': (790, 812),
    'C6H12_1266': (1254, 1278),
    'PS_1001': (993, 1008),
    'PS_1602': (1590, 1614),
    'cSi_520': (512, 528)
}
exclude_for_baseline = list(windows.values())

## Chargement d'un fichier et extraction des métriques (exemple SO₄²⁻)

In [ ]:
import numpy as np
p = data_dir/'SO4_example_50mM.csv'
arr = np.genfromtxt(p, delimiter=',', names=True)
x = arr['wavenumber_cm_1']
y = arr['intensity']
y_s = R.moving_average(y, w=11)
base,_ = R.poly_baseline(x, y_s, deg=3, exclude_windows=exclude_for_baseline)
y_corr = y_s - base
w = windows['SO4']
area = R.integrate_band(x, y_corr, w)
cent = R.centroid(x, y_corr, w)
w50 = R.fwhm_estimate(x, y_corr, w)
area, cent, w50

### Visualisation rapide

In [ ]:
plt.figure(figsize=(8,3))
plt.plot(x, y, label='Brut', alpha=0.6)
plt.plot(x, y_s, label='Lissé', alpha=0.8)
plt.plot(x, base, label='Baseline', ls='--')
plt.plot(x, y_corr, label='Corrigé')
a,b = windows['SO4']
plt.axvspan(a,b, alpha=0.2, label='Fenêtre SO4')
plt.legend(); plt.xlabel('cm$^{-1}$'); plt.ylabel('a.u.'); plt.tight_layout()
plt.show()

## Courbe d'étalonnage (aire de bande vs concentration) — gabarit

In [ ]:
def fit_linear(cs, areas):
    cs = np.asarray(cs); areas = np.asarray(areas)
    A = np.c_[cs, np.ones_like(cs)]
    k, b = np.linalg.lstsq(A, areas, rcond=None)[0]
    pred = k*cs + b
    rmse = np.sqrt(np.mean((areas-pred)**2))
    return k, b, rmse

cs = []; areas = []
for fp in [data_dir/'SO4_example_50mM.csv']:
    arr = np.genfromtxt(fp, delimiter=',', names=True)
    c = arr['concentration_mM'][0]
    x = arr['wavenumber_cm_1']; y = arr['intensity']
    y_s = R.moving_average(y, w=11)
    base,_ = R.poly_baseline(x, y_s, deg=3, exclude_windows=list(windows.values()))
    y_corr = y_s - base
    a = R.integrate_band(x, y_corr, windows['SO4'])
    cs.append(float(c)); areas.append(a)
k,b,rmse = fit_linear(cs, areas)
k,b,rmse

### Tracé de calibration (gabarit)

In [ ]:
plt.figure(figsize=(4,3))
plt.scatter(cs, areas, label='aires')
xx = np.linspace(0, max(cs+[1]), 50)
plt.plot(xx, k*xx+b, label=f'y={k:.3g}x+{b:.3g}')
plt.xlabel('Concentration (mM)'); plt.ylabel('Aire de bande (corrigée)')
plt.title('SO4²⁻ calibration (gabarit)'); plt.legend(); plt.tight_layout(); plt.show()

## Contrôles solides/liquides (c-Si, PS, cyclohexane)

In [ ]:
import json
def qc_peak(file, band_key):
    arr = np.genfromtxt(file, delimiter=',', names=True)
    x = arr['wavenumber_cm_1']; y = arr['intensity']
    y_s = R.moving_average(y, w=11)
    base,_ = R.poly_baseline(x, y_s, deg=3, exclude_windows=list(windows.values()))
    y_corr = y_s - base
    c = R.centroid(x, y_corr, windows[band_key])
    f = R.fwhm_estimate(x, y_corr, windows[band_key])
    return float(c), float(f)

c520,f520 = qc_peak(data_dir/'cSi_example.csv', 'cSi_520')
c1001,f1001 = qc_peak(data_dir/'polystyrene_example.csv', 'PS_1001')
c1602,f1602 = qc_peak(data_dir/'polystyrene_example.csv', 'PS_1602')
c801,f801 = qc_peak(data_dir/'cyclohexane_example.csv', 'C6H12_801')
c1266,f1266 = qc_peak(data_dir/'cyclohexane_example.csv', 'C6H12_1266')
qc = {
 'cSi_520': {'centroid':c520,'FWHM':f520},
 'PS_1001': {'centroid':c1001,'FWHM':f1001},
 'PS_1602': {'centroid':c1602,'FWHM':f1602},
 'C6H12_801': {'centroid':c801,'FWHM':f801},
 'C6H12_1266': {'centroid':c1266,'FWHM':f1266},
}
qc

## Export JSON

In [ ]:
out = (base/'figures'/'calibration_results.json')
with open(out,'w',encoding='utf-8') as f:
    json.dump({'qc': qc, 'fit': {'k': float(k), 'b': float(b), 'rmse': float(rmse)}}, f, indent=2)
out